In [1]:
# COUNTRY LEVEL - Simple Losses Ranking
from pathlib import Path
import pandas as pd
import geopandas as gpd

# Load input file
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Losses_Analysis\aggregated_loss_per_nws_code.gpkg")
gdf_src = gpd.read_file(input_path)

# Ensure required columns
if "VHLH_AL_E_WR" not in gdf_src.columns:
    raise KeyError("Column 'VHLH_AL_E_WR' not found in losses input.")

# Prepare data
gdf_src["VHLH_AL_E_WR"] = pd.to_numeric(gdf_src["VHLH_AL_E_WR"], errors="coerce")

# ✅ COUNTRY-WIDE ranking: assign max rank to missing or zero values
def custom_rank_countrywide(series):
    ranks = series.rank(ascending=False, method="min")
    ranks[series.isna() | (series == 0)] = len(series)
    return ranks

gdf_src["rk_VHLH_AL_E_WR"] = custom_rank_countrywide(gdf_src["VHLH_AL_E_WR"])

# Sort for readability
gdf_loss_ranked = (
    gdf_src
      .sort_values(["rk_VHLH_AL_E_WR", "VHLH_AL_E_WR"], ascending=[True, False], kind="mergesort")
      .reset_index(drop=True)
)

# Output paths
base_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
base_dir.mkdir(parents=True, exist_ok=True)
gpkg_path = base_dir / "Simple_Losses_ranking_country_lvl.gpkg"
shp_path  = base_dir / "Simple_Losses_ranking_country_lvl.shp"

# Save GPKG
gdf_loss_ranked.to_file(gpkg_path, layer="ranked_rows", driver="GPKG")

# Save SHP (shorten field names)
gdf_loss_ranked_shp = gdf_loss_ranked.copy()
gdf_loss_ranked_shp["rk_VHLH_AL_E_WR"] = gdf_loss_ranked_shp["rk_VHLH_AL_E_WR"].fillna(-1).astype("int32")
gdf_loss_ranked_shp = gdf_loss_ranked_shp.rename(columns={"rk_VHLH_AL_E_WR": "rk_VHLH"})

# Clean old shapefile sidecars
def delete_shapefile(path: Path):
    for ext in [".shp", ".shx", ".dbf", ".prj", ".cpg", ".shp.xml", ".qpj"]:
        p = path.with_suffix(ext)
        if p.exists():
            p.unlink()

delete_shapefile(shp_path)
gdf_loss_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")

print(f"Wrote country-level:\n- {gpkg_path}\n- {shp_path}")

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('ranked_rows')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('ranked_rows')) failed: unable to open database file"
C:\Users\gunaratn\AppData\Local\Temp\ipykernel_9072\3773945281.py:54: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_loss_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")


Wrote country-level:
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking_country_lvl.gpkg
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking_country_lvl.shp


In [2]:
# COUNTRY LEVEL - Damage Risk Ranking
from pathlib import Path
import pandas as pd
import geopandas as gpd

# Input
input_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damages_Aggregated_by_Schakels.gpkg")
gdf_src = gpd.read_file(input_path)

# Calculate metrics
gdf_src["dam_per_m"] = (
    (gdf_src["total_damage"] / gdf_src["total_length"])
    .where(gdf_src["total_length"].notna() & (gdf_src["total_length"] != 0))
)

gdf_src["fraction_flooded"] = (
    (gdf_src["flooded_length"] / gdf_src["total_length"])
    .where(gdf_src["total_length"].notna() & (gdf_src["total_length"] != 0))
)

# ✅ COUNTRY-WIDE ranking: assign max rank to missing or zero values
def custom_rank_countrywide(series):
    ranks = series.rank(ascending=False, method="min")
    ranks[series.isna() | (series == 0)] = len(series)
    return ranks

gdf_src["rank_frfl"] = custom_rank_countrywide(gdf_src["fraction_flooded"])
gdf_src["rk_d_m"] = custom_rank_countrywide(gdf_src["dam_per_m"])

# Prepare ranked output
required = ["Areas_name", "rank_frfl", "rk_d_m", "dam_per_m", "NWSCODE", "total_damage", "fraction_flooded",
            "total_length", "flooded_length", "tunnel_length_sum", "bridge_length_sum", "geometry"]
missing = [c for c in required if c not in gdf_src.columns]
if missing:
    raise KeyError(f"Missing in gdf_src: {missing}")

gdf_ranked = (
    gdf_src.loc[:, required]
            .sort_values(["rk_d_m", "dam_per_m"], ascending=[True, False], kind="mergesort")
            .reset_index(drop=True)
)

# Output paths
base_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
base_dir.mkdir(parents=True, exist_ok=True)
gpkg_path = base_dir / "Damage_risk_ranking_schakels_country_lvl.gpkg"
shp_path  = base_dir / "Damage_risk_ranking_schakels_country_lvl.shp"

# Save GPKG
gdf_ranked.to_file(gpkg_path, layer="ranked_rows", driver="GPKG")

# Save SHP (shorten field names)
gdf_ranked_shp = gdf_ranked.rename(columns={
    "total_damage": "tot_dam",
    "fraction_flooded": "frac_fld",
    "rk_d_m": "rk_d_m",
    "rank_frfl": "rk_frfl"
}).copy()

# Convert ranks to integer for SHP
gdf_ranked_shp["rk_d_m"] = gdf_ranked_shp["rk_d_m"].astype("Int64")
gdf_ranked_shp["rk_frfl"] = gdf_ranked_shp["rk_frfl"].astype("Int64")

gdf_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")

print(f"Wrote country-level:\n- {gpkg_path}\n- {shp_path}")


C:\Users\gunaratn\AppData\Local\Temp\ipykernel_9072\602357586.py:64: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf_ranked_shp.to_file(shp_path, driver="ESRI Shapefile")


Wrote country-level:
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damage_risk_ranking_schakels_country_lvl.gpkg
- P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Damage_risk_ranking_schakels_country_lvl.shp


In [5]:
print(losses_rank.columns)

Index(['NWSCODE', 'Length_schakel', 'NWSNAAM', 'VHLH_AL_E_WR', 'VHLH_L1_E_WR',
       'VHLH_L2_E_WR', 'VHLH_L3_E_WR', 'F_EV2_ma', 'F_EV1_me', 'AL_E_WR',
       'getroffen_personen_perdag', 'getroffen_vracht_L2+L3_E_WR_perdag',
       'vracht_op_schakel_totaal', 'personen_op_schakel_totaal',
       'VOT_L1_gemiddeld', 'VOT_L2L3_gemiddeld', 'VOT_total', 'VOT_Total_day',
       'rk_VHLH_AL_E_WR', 'geometry'],
      dtype='object')


In [8]:
# COUNTRY LEVEL - Combined Risk Ranking
import geopandas as gpd
import pandas as pd
from pathlib import Path

# Load losses ranking (country-level)
losses_rank_path = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Simple_Losses_ranking_country_lvl.gpkg")
losses_rank = gpd.read_file(losses_rank_path, driver="GPKG")

# Paths for (country-level) damages
root_dir = Path(r"P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs")
gpkg_path = root_dir / "Damage_risk_ranking_schakels_country_lvl.gpkg"

# Read damages
damages_rank = gpd.read_file(gpkg_path, driver="GPKG")

# Merge VHLH and damages (keep Area for reference only)
vhlh_df = losses_rank[['NWSCODE', 'NWSNAAM', "rk_VHLH_AL_E_WR", 'F_EV2_ma',"VHLH_AL_E_WR"]].copy()
damage_df = damages_rank[['NWSCODE', 'rk_d_m', 'flooded_length',"fraction_flooded", "dam_per_m",'tunnel_length_sum', 'geometry']].copy()

df = pd.merge(damage_df, vhlh_df, on=['NWSCODE'], how='inner')

# Convert back to GeoDataFrame
gdf = gpd.GeoDataFrame(df, geometry='geometry', crs=damages_rank.crs)

# ✅ Invert ranks (COUNTRY-WIDE)
max_vhlh = gdf["rk_VHLH_AL_E_WR"].max()
max_damage = gdf['rk_d_m'].max()
gdf['inv_VHLH'] = (max_vhlh + 1) - gdf['rk_VHLH_AL_E_WR']
gdf['inv_Damage'] = (max_damage + 1) - gdf['rk_d_m']

# ✅ Tunnel ranking: assign max rank to missing or zero values (COUNTRY-WIDE)
def custom_rank(series):
    ranks = series.rank(ascending=False, method="min")
    ranks[series.isna() | (series == 0)] = len(series)
    return ranks

gdf['tunnel_rank'] = custom_rank(gdf['tunnel_length_sum'])
max_tunnel = gdf['tunnel_rank'].max()
gdf['inv_Tunnel'] = (max_tunnel + 1) - gdf['tunnel_rank']

# ✅ Rank flooded_length (COUNTRY-WIDE)
gdf['flooded_rank'] = gdf['flooded_length'].rank(method='dense', ascending=False)

# ✅ Compute weighted score (Damage, VHLH, Tunnel)
weights = {'Damage': 0.3, 'VHLH': 0.5, 'Tunnels': 0.2}
gdf['Total'] = (
    gdf['inv_VHLH'] * weights['VHLH'] +
    gdf['inv_Damage'] * weights['Damage'] +
    gdf['inv_Tunnel'] * weights['Tunnels']
)

gdf = gdf.sort_values('Total', ascending=False).reset_index(drop=True)

# ✅ Assign final rank (COUNTRY-WIDE)
gdf['Final_rank'] = gdf['Total'].rank(method='min', ascending=False).astype(int)

# ✅ Keep flooded_length and flooded_rank in final output
columns_to_keep = [
    'NWSCODE', 'NWSNAAM','F_EV2_ma',"fraction_flooded", "dam_per_m","VHLH_AL_E_WR","rk_VHLH_AL_E_WR", 'rk_d_m',
    'flooded_length', 'flooded_rank', 'tunnel_length_sum',
    'inv_VHLH', 'inv_Damage', 'inv_Tunnel', 'Total', 'Final_rank', 'geometry'
]
gdf = gdf[columns_to_keep]

# Save output
out_path = root_dir / "Combined_Risk_Ranking_country_lvl_ver03_newWeights.gpkg"
gdf.to_file(out_path, driver="GPKG")

print(f"Saved country-level ranking to {out_path}")
print(f"\nTop 10 highest risk locations:")
print(gdf[['NWSCODE', 'NWSNAAM', 'Final_rank', 'Total']].head(10))

CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking_country_lvl_ver03_newWeights')) failed: unable to open database file"

Exception ignored in: 'fiona._shim.gdal_flush_cache'
Traceback (most recent call last):
  File "fiona\_err.pyx", line 201, in fiona._err.GDALErrCtxManager.__exit__
fiona._err.CPLE_AppDefinedError: b"sqlite3_exec(UPDATE gpkg_contents SET last_change = strftime('%Y-%m-%dT%H:%M:%fZ','now') WHERE lower(table_name) = lower('Combined_Risk_Ranking_country_lvl_ver03_newWeights')) failed: unable to open database file"


Saved country-level ranking to P:\bovenregionale-stresstest-hwn\Analysis\Merged_Outputs\Combined_Risk_Ranking_country_lvl_ver03_newWeights.gpkg

Top 10 highest risk locations:
    NWSCODE                                            NWSNAAM  Final_rank  \
0  004-0050  A4 KP Burgerveen - Leiden Aansluiting N11 (dee...           1   
1  028-0010                  A28 KP Rijnsweerd - KP Hoevelaken           2   
2  004-0090            A4 Kruithuisweg (N470) - KP Kethelplein           3   
3  010-0030                  A10 KP De Nieuwe Meer - KP Amstel           4   
4  059-0105                       A59 KP Hooipolder - KP Empel           5   
5  002-0090                A2 KP Empel - KP Hintham - KP Vught           6   
6  020-0050                   A20 KP Terbregseplein - KP Gouwe           7   
7  031-1010                    A31 KP Zurich - KP Werpsterhoek           8   
8  011-0010                       N11 Leiden - Aansluiting A12           9   
9  028-1040                     A28 KP Assen